# Secret-Learn Quick Start Tutorial

This notebook demonstrates how to use **Secret-Learn** for privacy-preserving machine learning with SecretFlow.

## Overview

Secret-Learn provides **573 implementations** (191 algorithms × 3 privacy modes):

| Mode | Prefix | Description | Use Case |
|------|--------|-------------|----------|
| **Federated Learning** | `FL` | Horizontal data partitioning, local training with secure aggregation | Multiple organizations with similar data schemas |
| **Secret Sharing** | `SS` | Multi-party computation via SPU, encrypted computation | Maximum security, small-medium datasets |
| **Split Learning** | `SL` | Model layer splitting, only activations exchanged | Vertical data partitioning, large models |

## Prerequisites

```bash
pip install secret-learn secretflow
```

## 1. Environment Setup

First, let's import the necessary libraries and initialize SecretFlow.

In [ ]:
import numpy as np
import random

# SecretFlow imports
import secretflow as sf
import secretflow.distributed as sfd
from secretflow.data import FedNdarray, PartitionWay
from secretflow.device.driver import reveal
from secretflow.distributed.const import DISTRIBUTION_MODE

print("✓ Imports successful")
print(f"  SecretFlow version: {sf.__version__}")

In [ ]:
# Initialize SecretFlow with random ports (avoid conflicts)
base_port = random.randint(10000, 60000)

cluster_config = {
    'parties': {
        'alice': {'address': f'localhost:{base_port}', 'listen_addr': f'0.0.0.0:{base_port}'},
        'bob': {'address': f'localhost:{base_port+1}', 'listen_addr': f'0.0.0.0:{base_port+1}'},
        'carol': {'address': f'localhost:{base_port+2}', 'listen_addr': f'0.0.0.0:{base_port+2}'},
    },
    'self_party': 'alice'
}

# Initialize with PRODUCTION mode
sfd.init(DISTRIBUTION_MODE.PRODUCTION, cluster_config=cluster_config)

# Create SPU device for Secret Sharing mode
spu_config = sf.utils.testing.cluster_def(
    parties=['alice', 'bob', 'carol'],
    runtime_config={'protocol': 'ABY3', 'field': 'FM64'}
)
spu = sf.SPU(spu_config)

# Create PYU devices
alice = sf.PYU('alice')
bob = sf.PYU('bob')
carol = sf.PYU('carol')

print("✓ SecretFlow initialized")
print(f"  Parties: alice, bob, carol")
print(f"  Base port: {base_port}")

## 2. Prepare Sample Data

We'll create synthetic data and partition it across parties for federated computation.

In [ ]:
# Generate sample data
np.random.seed(42)
n_samples = 500
n_features = 12

X = np.random.randn(n_samples, n_features).astype(np.float32)
y = (X[:, 0] + X[:, 1] * 2 + np.random.randn(n_samples) * 0.1).astype(np.float32)
y_class = (y > 0).astype(np.int32)  # Binary classification target

# Vertical partitioning (each party has different features)
X_alice = X[:, :4]   # Alice has features 0-3
X_bob = X[:, 4:8]    # Bob has features 4-7
X_carol = X[:, 8:]   # Carol has features 8-11

print(f"✓ Data generated")
print(f"  Total: {n_samples} samples × {n_features} features")
print(f"  Alice: {X_alice.shape}, Bob: {X_bob.shape}, Carol: {X_carol.shape}")

In [ ]:
# Create federated data structures
fed_X = FedNdarray(
    partitions={
        alice: alice(lambda x: x)(X_alice),
        bob: bob(lambda x: x)(X_bob),
        carol: carol(lambda x: x)(X_carol),
    },
    partition_way=PartitionWay.VERTICAL
)

fed_y = FedNdarray(
    partitions={alice: alice(lambda x: x)(y)},
    partition_way=PartitionWay.HORIZONTAL
)

fed_y_class = FedNdarray(
    partitions={alice: alice(lambda x: x)(y_class)},
    partition_way=PartitionWay.HORIZONTAL
)

print("✓ Federated data created")

## 3. Federated Learning (FL) Example

Federated Learning trains models locally on each party's data and securely aggregates the results.
Each party keeps their raw data private.

In [ ]:
import time
from secretlearn.federated_learning.linear_models.linear_regression import FLLinearRegression

# Create device dictionary for FL
devices = {"alice": alice, "bob": bob, "carol": carol}

# Train FL model
print("Training FLLinearRegression...")
start = time.time()

fl_model = FLLinearRegression(devices)
fl_model.fit(fed_X, fed_y)

elapsed = time.time() - start
print(f"✓ FL training completed in {elapsed*1000:.2f}ms")

## 4. Secret Sharing (SS) Example

Secret Sharing uses Multi-Party Computation (MPC) via SPU.
Data is encrypted and split into shares - no party sees the complete data.

In [ ]:
from secretlearn.secret_sharing.decomposition.pca import SSPCA

# Train SS model using SPU
print("Training SSPCA (Secret Sharing PCA)...")
start = time.time()

ss_model = SSPCA(spu)
ss_model.fit(fed_X)  # Unsupervised - no y needed

elapsed = time.time() - start
print(f"✓ SS training completed in {elapsed*1000:.2f}ms")

## 5. Split Learning (SL) Example

Split Learning splits the model across parties.
Only intermediate activations are exchanged, not raw data or complete models.

In [ ]:
from secretlearn.split_learning.clustering.kmeans import SLKMeans

# Train SL model
print("Training SLKMeans (Split Learning KMeans)...")
start = time.time()

sl_model = SLKMeans(devices, n_clusters=3)
sl_model.fit(fed_X)  # Unsupervised - no y needed

elapsed = time.time() - start
print(f"✓ SL training completed in {elapsed*1000:.2f}ms")

## 6. Cleanup

Always shutdown SecretFlow when done to release resources.

In [ ]:
sf.shutdown(wait_for_sending=False)
print("✓ SecretFlow shutdown complete")

## Summary

You've learned how to use all three privacy modes in Secret-Learn:

| Mode | Algorithm Used | Key Feature |
|------|---------------|-------------|
| **FL** | `FLLinearRegression` | Local training + secure aggregation |
| **SS** | `SSPCA` | MPC via SPU, encrypted computation |
| **SL** | `SLKMeans` | Model splitting, activation exchange |

### Next Steps

Explore more algorithms:
- `examples/federated_learning/` - 193 FL examples
- `examples/secret_sharing/` - 193 SS examples
- `examples/split_learning/` - 193 SL examples

### Import Pattern

```python
# FL Mode
from secretlearn.federated_learning.{category}.{algorithm} import FL{Algorithm}

# SS Mode
from secretlearn.secret_sharing.{category}.{algorithm} import SS{Algorithm}

# SL Mode
from secretlearn.split_learning.{category}.{algorithm} import SL{Algorithm}
```